In [1]:
import torch
import numpy as np
import pandas as pd

In [2]:
torch.cuda.is_available()

True

##### import data

In [3]:
df = pd.read_csv("../data/emotions_data.csv")
df.head()

,text,label
0,Just got back from seeing @GaryDelaney in Burs...,joy
1,Oh dear an evening of absolute hilarity I don'...,joy
2,Been waiting all week for this game ❤️❤️❤️ #ch...,joy
3,"@gardiner_love : Thank you so much, Gloria! Yo...",joy
4,I feel so blessed to work with the family that...,joy


#### preprocess

##### handle special chars

In [4]:
import re

df['text'] = df['text'].apply(lambda row: re.sub(r'[^\w\s]', '', row))
df.head()

,text,label
0,Just got back from seeing GaryDelaney in Bursl...,joy
1,Oh dear an evening of absolute hilarity I dont...,joy
2,Been waiting all week for this game cheer fri...,joy
3,gardiner_love Thank you so much Gloria Youre ...,joy
4,I feel so blessed to work with the family that...,joy


##### encode labels

In [5]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

df['label'] = pd.Series(le.fit_transform(df['label'])) # le.fit_transform gives numpy array
df.head()

,text,label
0,Just got back from seeing GaryDelaney in Bursl...,2
1,Oh dear an evening of absolute hilarity I dont...,2
2,Been waiting all week for this game cheer fri...,2
3,gardiner_love Thank you so much Gloria Youre ...,2
4,I feel so blessed to work with the family that...,2


In [6]:
le.classes_

array(['anger', 'fear', 'joy', 'sadness'], dtype=object)

##### split data

In [7]:
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(df, test_size = 0.3, stratify = df['label'], random_state = 0) # stratify: each class keeps same proportion in train and test set

##### convert to hugging face dataset

> conversion is needed to get better features and compatibility

In [8]:
from datasets import Dataset

train_data = Dataset.from_pandas(train_data.reset_index(drop = True))
test_data = Dataset.from_pandas(test_data.reset_index(drop = True))

print(train_data, train_data[0], sep = '\n\n')

Dataset({
    features: ['text', 'label'],
    num_rows: 2529
})

{'text': 'You have to find a way to top yourself glee', 'label': 2}


#### tokenize

In [9]:
from transformers import XLNetTokenizer

In [10]:
model_name = 'xlnet-base-cased'

In [11]:
tokenizer = XLNetTokenizer.from_pretrained(model_name)

In [12]:
def tokenize_function(example):
    return tokenizer(example['text'], padding = 'max_length', truncation = True, max_length = 32)

train_token_data = train_data.map(tokenize_function, batched = True) # batched = True, for fast batch tokenization
test_token_data = test_data.map(tokenize_function, batched = True)

Map:   0%|          | 0/2529 [00:00<?, ? examples/s]

Map:   0%|          | 0/1084 [00:00<?, ? examples/s]

In [13]:
train_token_data.set_format(type = 'torch') # convert numericals fields to tensor
test_token_data.set_format(type = 'torch')

In [14]:
print(train_token_data, train_token_data[0], sep = '\n\n')

Dataset({
    features: ['text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 2529
})

{'text': 'You have to find a way to top yourself glee', 'label': tensor(2), 'input_ids': tensor([    5,     5,     5,     5,     5,     5,     5,     5,     5,     5,
            5,     5,     5,     5,     5,     5,     5,     5,     5,     5,
          201,    47,    22,   278,    24,   162,    22,   310,  1804, 26774,
            4,     3]), 'attention_mask': tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1])}


#### fine-tune XLNet

In [15]:
from transformers import XLNetForSequenceClassification, TrainingArguments, Trainer

In [16]:
model = XLNetForSequenceClassification.from_pretrained(model_name, num_labels = len(le.classes_), id2label = {ind: label for ind, label in enumerate(le.classes_)})

Loading weights:   0%|          | 0/206 [00:00<?, ?it/s]

XLNetForSequenceClassification LOAD REPORT from: xlnet-base-cased
Key                             | Status     | 
--------------------------------+------------+-
lm_loss.bias                    | UNEXPECTED | 
lm_loss.weight                  | UNEXPECTED | 
logits_proj.weight              | MISSING    | 
sequence_summary.summary.weight | MISSING    | 
logits_proj.bias                | MISSING    | 
sequence_summary.summary.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [17]:
training_args = TrainingArguments(
    output_dir = "../models/xlnet_emotion/", # save model checkpoints (saved weights)
    num_train_epochs = 10,
    per_device_train_batch_size = 64,
    per_device_eval_batch_size = 64,
    eval_strategy = 'epoch',
    save_strategy = 'epoch', # need to save to load best model
    save_total_limit = 2, # limit no. of saved checkpoints
    load_best_model_at_end=True, # looks at all saved checkpoints and picks the best one
    metric_for_best_model = 'f1',
    greater_is_better = True, # maximize the metric
    fp16 = torch.cuda.is_available() # enable mixed precision for faster training
)

In [18]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    
    preds = np.argmax(logits, axis = 1) # for each input: index of label with max logit (i.e. the label itself)

    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds, average = 'weighted') # 'average' needed for multiclass
    }

> trainer automatically moves model to gpu or cpu, whichever is available

In [19]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_token_data,
    eval_dataset = test_token_data,
    compute_metrics = compute_metrics
)

> resume_from_checkpoint: complete previous training

In [20]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,1.045325,0.580258,0.564189
2,No log,0.570962,0.791513,0.790322
3,No log,0.547178,0.809041,0.808103
4,No log,0.561019,0.825646,0.826060
5,No log,0.628393,0.822878,0.822144
6,No log,0.661021,0.833026,0.832675
7,No log,0.650543,0.847786,0.848030
8,No log,0.710222,0.853321,0.852967
9,No log,0.718336,0.847786,0.847612
10,No log,0.739947,0.845018,0.844836


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=400, training_loss=0.3348991394042969, metrics={'train_runtime': 144.5782, 'train_samples_per_second': 174.923, 'train_steps_per_second': 2.767, 'total_flos': 450296359902720.0, 'train_loss': 0.3348991394042969, 'epoch': 10.0})

##### save model and tokenizer

In [21]:
model_directory = "../models/xlnet_emotion/final/"

In [22]:
model.save_pretrained(model_directory)
tokenizer.save_pretrained(model_directory)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('../models/xlnet_emotion/final/tokenizer_config.json',
 '../models/xlnet_emotion/final/tokenizer.json')

#### predict on new input

In [23]:
text = "I am very happy that I got to travel to such beautiful places. I wish to continue this for the rest of my life!"

In [24]:
tokenizer = XLNetTokenizer.from_pretrained(model_directory)
model = XLNetForSequenceClassification.from_pretrained(model_directory)

Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

In [25]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # no Trainer here, so we need to specify the device

In [26]:
def emotion_classifier(text, model):
    text = re.sub(r'[^\w\s]', '', text)

    input_data = tokenizer(text, return_tensors = 'pt', padding = 'max_length', truncation = True, max_length = 32)

    # model and input must be on the same device (cpu or gpu)
    model.to(device)
    input_data = {key: val.to(device) for key, val in input_data.items()} # move tensors to selected device

    with torch.no_grad():
        output = model(**input_data)

    logits = output.logits
    pred = torch.argmax(logits, dim = 1).item()

    label = model.config.id2label[pred]

    return label

In [27]:
label = emotion_classifier(text, model)
print(label)

joy
